# 02 - Preprocessing (shared)

This notebook builds the **shared** representation that *every* model in the project uses: Popularity, Markov, FPMC, GRU4Rec and SASRec. All the logic lives in `src/preprocessing.py`; this notebook only calls it, documents each decision, and shows the result.

**Why shared:** identical sequences, id mappings and filtering are what make the final comparison table apples-to-apples. If two teammates preprocess differently, their scores are not comparable even if both models are correct.

In [ ]:
# --- Setup: make the project root importable, locate the local data/ folder ---
import os, sys

def find_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    for _ in range(6):
        if os.path.exists(os.path.join(d, 'src', 'preprocessing.py')):
            return d
        d = os.path.dirname(d)
    return os.path.abspath(os.path.join(os.getcwd(), '..'))  # fallback: notebooks/ -> root

ROOT = find_root()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
DATA_DIR = os.path.join(ROOT, 'data')
PROC_DIR = os.path.join(ROOT, 'data', 'processed')
print('ROOT     =', ROOT)
print('DATA_DIR =', DATA_DIR)

In [ ]:
# --- Colab only: uncomment to fetch the CSVs into data/ (dev convenience) ---
# The graded pipeline never touches a URL; this cell just populates the local
# data/ folder when running on a fresh Colab machine.
# !python {os.path.join(ROOT, 'scripts', 'download_data.py')} --out {DATA_DIR}

In [ ]:
# --- Offline smoke test: proves the pipeline logic works on synthetic data ---
# (runs even before the real CSVs are present)
import subprocess
res = subprocess.run([sys.executable, '-m', 'src.preprocessing'], cwd=ROOT,
                     capture_output=True, text=True)
print(res.stdout or res.stderr)

## Decision 1 - Minimum trip length (drop trips < 4 stops, TRAIN only)

Every trip in the held-out **test** set has at least four stops (the fourth being the masked city we must predict). To keep training and evaluation on the same footing we drop training trips shorter than four stops. Length-1 trips are dropped in any case: with a single city there is no next-city to predict.

*Trade-off:* discarding 2-3 stop trips removes some training signal that the embedding models (FPMC, SASRec) could otherwise exploit. We accept this for consistency with the test distribution and keep `min_trip_len` a parameter so the choice can be revisited as an ablation. The test set is used exactly as shipped and is **never** filtered.

## Decision 2 - Collapse consecutive duplicate cities

If a traveller books the same city in two back-to-back rows, that is a longer stay, not a move to a new destination. We collapse such consecutive duplicates so the models learn genuine city-to-city transitions rather than a trivial self-loop that would inflate every next-item metric.

## Decision 3 - Categorical encoding and the synthetic persona

`city_id`, `user_id`, `hotel_country`, `booker_country`, `device_class` and `affiliate_id` are each mapped to contiguous integers using mappings built **only from the training set**; index 0 is reserved for padding and for any value unseen in training. This guarantees every model interprets an id the same way.

Because nearly every `user_id` appears in a single trip, raw user identity carries almost no transferable preference signal. We therefore also build a **persona** id = (`booker_country`, `device_class`), a coarse profile that recurs across many trips. FPMC is later run with both to show the difference - a headline finding for the report.

## Decision 4 - Engineered per-step features

For each stop we record the **month** of check-in (1-12, for seasonality), the **length of stay** in nights, and the **trip position** (absolute index, a normalised 0-1 fraction, and a 5-bucket version). Position is included mainly for FPMC: unlike SASRec, which learns positional embeddings automatically, FPMC has no built-in notion of where in the trip a step sits.

In [ ]:
# --- Run the real pipeline on a 5k-trip subsample for fast, end-to-end dev ---
from src.preprocessing import preprocess

data = preprocess(data_dir=DATA_DIR, out_dir=PROC_DIR,
                  min_trip_len=4, sample_n_trips=5000, seed=42)
data['meta']

In [ ]:
# --- Sanity peek: one train trip and one test trip ---
print('n_cities (incl PAD/UNK):', data['n_cities'])
print('\nexample TRAIN trip:')
for k, v in data['train'][0].items():
    print(f'  {k:15s}: {v}')
print('\nexample TEST trip (cities = observed prefix, target = held-out city):')
for k, v in data['test'][0].items():
    print(f'  {k:15s}: {v}')

## Running the full dataset (for final report numbers)

Set `sample_n_trips=None` to use every training trip. The processed result is pickled to `data/processed/processed.pkl`; each model notebook loads it with `preprocessing.load_processed()` rather than re-running this step.

```python
data = preprocess(data_dir=DATA_DIR, out_dir=PROC_DIR, min_trip_len=4, sample_n_trips=None)
```

Preprocessing itself is CPU-only and fine locally. Note for later: **full-scale SASRec training should be run on a Colab GPU**, not locally.